In [ ]:
##PROJECT STATEMENT
#lets define the problem 
#The purpose of this project is to analyse two years of personal financial transactions to understand spending behaviour, identify major expense categories, measure savings performance, and develop a realistic savings and investment strategy

In [ ]:
#lets load the dataset

import pandas as pd

file_path = "data/transactions.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

In [ ]:
from pathlib import Path

print(Path.cwd())

In [ ]:
df=pd.read_excel(file_path)
df.head()

In [ ]:
#lets inpect the dataset 
print(df.shape)
print(df.columns)
print(df.info())
print(df.isnull().sum())

In [ ]:
#lets read both sheets correctly
wallet=pd.read_excel(
    file_path,
    sheet_name="Wallet Account Transactions",
    header=6
)
savings = pd.read_excel(
    file_path,
    sheet_name="Savings Account Transactions",
    header=6
)

In [ ]:
#lets the check the result
print(wallet.shape)
print(savings.shape)

display(wallet.head())
display(savings.head())

In [ ]:
#since we want to combine the two sheets, we should be able to identify the source accounts 
wallet["account_source"] = "Wallet"
savings["account_source"] = "Savings"

In [ ]:
#lets combine the tables 
transactions=pd.concat(
    [wallet,savings],
    ignore_index=True
)
print(transactions.shape)

In [ ]:
#lets rename the columns 
transactions = transactions.rename(
    columns={
        "Trans. Date": "transaction_datetime",
        "Value Date": "value_date",
        "Description": "description",
        "Debit(₦)": "debit",
        "Credit(₦)": "credit",
        "Balance After(₦)": "balance_after",
        "Channel": "channel",
        "Transaction Reference": "transaction_reference"
    }
)

print(transactions.columns)


In [ ]:
#Lets clean the monetary columns 
money_columns=["debit","credit","balance_after"]

for column in money_columns:
    transactions[column]=(
        transactions[column]
        .astype("string")
        .str.replace("₦","",regex=False)
        .str.replace(",","",regex=False)
        .str.strip()
        .replace({"--": pd.NA, "": pd.NA})
    )

    transactions[column]=pd.to_numeric(
        transactions[column],
        errors="coerce"
    )

In [ ]:
transactions[
    ["transaction_datetime", "value_date"]
].head()

In [ ]:
#lets clean the text columns 
text_columns = [
    "description",
    "channel",
    "transaction_reference",
    "account_source"
]

for column in text_columns:
    transactions[column] = (
        transactions[column]
        .astype("string")
        .str.strip()
    )

In [ ]:
#Lets create transaction type and amount 
import numpy as np
transactions["transaction_type"] = np.where(
    transactions["debit"].notna(),
    "Debit",
    "Credit"
)

#lets create an amount column 
transactions["amount"] = transactions["debit"].fillna(
    transactions["credit"]
)

#lets create a signed amount
transactions["signed_amount"] = (
    transactions["credit"].fillna(0)
    - transactions["debit"].fillna(0)
)

In [ ]:
#lets mark obvious internal transfers 
internal_descriptions = [
    "OWealth Withdrawal(Transaction Payment)",
    "Auto-save to OWealth Balance",
    "Spend & Save Deposit",
    "Spend & Save Withdrawal",
    "OWealth Deposit(from Spend & Save)",
    "OWealth Deposit(Transaction Refund)",
    "OWealth Deposit(from Fixed)",
    "OWealth Withdrawal"
]

transactions["is_internal_transfer"] = (
    transactions["description"]
    .isin(internal_descriptions)
)

In [ ]:
#lets mark transfers to myself 
transactions["potential_self_transfer"] = (
    transactions["description"]
    .str.contains(
        "YOUR ACCOUNT NAME",
        case=False,
        na=False
    )
)

In [ ]:
print(transactions["transaction_datetime"].dtype)

In [ ]:
#Lets change the format of the dates 

transactions["transaction_datetime"] = pd.to_datetime(
    transactions["transaction_datetime"],
    format="%d %b %Y %H:%M:%S",
    errors="coerce"
)

In [ ]:
#Add useful date columns

transactions["date"] = (
    transactions["transaction_datetime"].dt.date
)

transactions["year"] = (
    transactions["transaction_datetime"].dt.year
)

transactions["month"] = (
    transactions["transaction_datetime"].dt.month
)

transactions["month_name"] = (
    transactions["transaction_datetime"].dt.month_name()
)

transactions["year_month"] = (
    transactions["transaction_datetime"]
    .dt.to_period("M")
    .astype(str)
)

transactions["day_of_week"] = (
    transactions["transaction_datetime"].dt.day_name()
)

In [ ]:
#lets check for missing data
transactions.isna().sum()

#lets check for duplicated rows
transactions.duplicated().sum()


In [ ]:
#lets review the cleaned data
display(transactions.head())

print(transactions.info())
print(transactions.shape)

In [ ]:
#lets save the cleaned data 
cleaned_folder = Path("outputs")

cleaned_folder.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_file = cleaned_folder / "opay_transactions_cleaned.csv"

transactions.to_csv(
    cleaned_file,
    index=False
)

print(cleaned_file)
print(cleaned_file.exists())

In [ ]:
#Lets categorize the data 
def categorise_transaction(description):
    description = str(description).lower()

    if "bolt" in description or "uber" in description:
        return "Transportation"
    elif "mtn" in description or "airtel" in description:
        return "Airtime and Internet"
    elif "netflix" in description or "spotify" in description:
        return "Subscriptions"
    elif "electricity" in description:
        return "Utilities"
    else:
        return "Other"


transactions["category"] = (
    transactions["description"]
    .apply(categorise_transaction)
)

In [ ]:
transactions[
    ["description", "category"]
].head(20)

In [ ]:
#lets find every account i sent money to 
#select outgoing transfers 
sent_transfers = transactions[
    (transactions["account_source"] == "Wallet")
    & (transactions["debit"].notna())
    & (
        transactions["description"]
        .astype(str)
        .str.lower()
        .str.startswith("transfer to", na=False)
    )
].copy()


#lets extract the beneficiary details 
beneficiary_details = (
    sent_transfers["description"]
    .str.split("|", expand=True)
    .reindex(columns=range(3))
)


sent_transfers["beneficiary_name"] = (
    beneficiary_details[0]
    .str.replace(
        r"(?i)^Transfer to\s+",
        "",
        regex=True
    )
    .str.strip(" -")
    .str.upper()
)


sent_transfers["beneficiary_bank"] = (
    beneficiary_details[1]
    .str.strip()
)


sent_transfers["beneficiary_account"] = (
    beneficiary_details[2]
    .str.strip()
)

In [ ]:
#lets check the extracted information
sent_transfers[
    [
        "description",
        "beneficiary_name",
        "beneficiary_bank",
        "beneficiary_account",
        "debit"
    ]
].head(20)

In [ ]:
#lets group the transactions by account 
beneficiary_summary = (
    sent_transfers
    .groupby(
        [
            "beneficiary_name",
            "beneficiary_bank",
            "beneficiary_account"
        ],
        dropna=False
    )
    .agg(
        total_sent=("debit", "sum"),
        number_of_transfers=("debit", "count"),
        average_transfer=("debit", "mean"),
        largest_transfer=("debit", "max")
    )
    .sort_values(
        "total_sent",
        ascending=False
    )
    .reset_index()
)

beneficiary_summary.index = range(1, len(beneficiary_summary) + 1)

beneficiary_summary
beneficiary_summary.head(20)

In [ ]:
#we are trying to create a safe version that conceals account information

import pandas as pd

def mask_account(account):
    if pd.isna(account):
        return pd.NA

    account = str(account).strip()

    if len(account) <= 4:
        return "*" * len(account)

    return "*" * (len(account) - 4) + account[-4:]

In [ ]:
beneficiary_summary["masked_account"] = (
    beneficiary_summary["beneficiary_account"]
    .apply(mask_account)
)

In [ ]:
safe_beneficiary_summary = (
    beneficiary_summary
    .drop(columns="beneficiary_account")
    .rename(columns={
        "masked_account": "beneficiary_account"
    })
)

In [ ]:
display(safe_beneficiary_summary.head(20))

In [ ]:
#Lets rank the data based on highest monthly flow 
# Extract sender names
transactions["sender"] = (
    transactions["description"]
    .str.extract(
        r"(?i)^Transfer\s+from\s+([^|]+)",
        expand=False
    )
    .str.upper()
    .str.strip()
)

# Accounts belonging to you
own_names = [
    "YOUR ACCOUNT NAME"
]

# Select genuine incoming Wallet transfers
incoming_transfers = transactions[
    (transactions["account_source"] == "Wallet")
    & (transactions["credit"].notna())
    & (
        transactions["description"]
        .str.lower()
        .str.startswith("transfer from", na=False)
    )
    & (~transactions["sender"].isin(own_names))
].copy()

In [ ]:
#lets create the month column 
incoming_transfers["year_month"] = (
    incoming_transfers["transaction_datetime"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
#lets calculate and rank monthly flows

monthly_inflow = (
    incoming_transfers
    .groupby("year_month")
    .agg(
        total_inflow=("credit", "sum"),
        number_of_transfers=("credit", "count")
    )
    .sort_values(
        "total_inflow",
        ascending=False
    )
    .reset_index()
)

monthly_inflow.index=range(
    1,
    len(monthly_inflow)+1
)

In [ ]:
#lets display the result 
display(
    monthly_inflow.style.format({
        "total_inflow": "₦{:,.2f}"
    })
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter


# Convert the transaction date column to datetime
transactions["transaction_datetime"] = pd.to_datetime(
    transactions["transaction_datetime"],
    errors="coerce"
)


# Convert the credit column to numbers
transactions["credit"] = pd.to_numeric(
    transactions["credit"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("₦", "", regex=False)
    .str.strip(),
    errors="coerce"
).fillna(0)


# Calculate total income received each month
monthly_income = (
    transactions
    .dropna(subset=["transaction_datetime"])
    .sort_values("transaction_datetime")
    .set_index("transaction_datetime")["credit"]
    .resample("MS")
    .sum()
    .rename("total_income")
    .reset_index()
)


# Find the month with the highest income
highest_index = monthly_income["total_income"].idxmax()
highest_month = monthly_income.loc[highest_index]


# Create the line chart
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    monthly_income["transaction_datetime"],
    monthly_income["total_income"],
    color="royalblue",
    linewidth=2.5,
    marker="o",
    markersize=6
)


# Highlight the month with the highest income
ax.scatter(
    highest_month["transaction_datetime"],
    highest_month["total_income"],
    color="red",
    s=100,
    zorder=3
)

ax.annotate(
    f'Highest: ₦{highest_month["total_income"]:,.0f}',
    xy=(
        highest_month["transaction_datetime"],
        highest_month["total_income"]
    ),
    xytext=(0, 30),
    textcoords="offset points",
    ha="center",
    fontweight="bold",
    arrowprops={
        "arrowstyle": "->",
        "color": "black"
    }
)


# Add titles and labels
ax.set_title(
    "Monthly Income Trend",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Month")
ax.set_ylabel("Total income")


# Display the y-axis values as naira
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, position: f"₦{value:,.0f}")
)


# Display dates as month and year
ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=1)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%b %Y")
)


# Add grid lines and rotate the dates
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#lets plot spending against income 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

# Ensure the date column is stored as datetime
transactions["transaction_datetime"] = pd.to_datetime(
    transactions["transaction_datetime"],
    errors="coerce"
)

description = (
    transactions["description"]
    .astype("string")
    .str.strip()
)

# Identify obvious internal savings movements
internal_transfer = description.str.contains(
    r"OWealth|Spend & Save|Auto-save",
    case=False,
    regex=True,
    na=False
)

# Identify transfers involving your own named account
self_inflow = description.str.contains(
    r"^Transfer\s+from\s+YOUR ACCOUNT NAME\b",
    case=False,
    regex=True,
    na=False
)

self_outflow = description.str.contains(
    r"^Transfer\s+to\s+YOUR ACCOUNT NAME\b",
    case=False,
    regex=True,
    na=False
)

# Identify incoming transfers
transfer_received = (
    description
    .str.lower()
    .str.startswith("transfer from", na=False)
)

# Select genuine incoming transfers
income_transactions = transactions[
    (transactions["account_source"] == "Wallet")
    & (transactions["credit"].notna())
    & transfer_received
    & (~self_inflow)
].copy()

# Select spending transactions
spending_transactions = transactions[
    (transactions["account_source"] == "Wallet")
    & (transactions["debit"].notna())
    & (~internal_transfer)
    & (~self_outflow)
].copy()

In [ ]:
#lets calculate the totals
monthly_income = (
    income_transactions
    .set_index("transaction_datetime")
    ["credit"]
    .resample("MS")
    .sum()
    .rename("income")
)

monthly_spending = (
    spending_transactions
    .set_index("transaction_datetime")
    ["debit"]
    .resample("MS")
    .sum()
    .rename("spending")
)

monthly_comparison = pd.concat(
    [monthly_income, monthly_spending],
    axis=1
).fillna(0).reset_index()

display(monthly_comparison)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter


# Enter the exact column names from month_comparison
month_column = "transaction_datetime"
income_column = "income"
outcome_column = "spending"   # Change to "outcome" or "outflow" if needed


# Create a copy for the visualisation
plot_data = monthly_comparison.copy()


# Convert the month column to a date
plot_data[month_column] = pd.to_datetime(
    plot_data[month_column],
    errors="coerce"
)


# Convert income and spending to numbers
for column in [income_column, outcome_column]:
    plot_data[column] = pd.to_numeric(
        plot_data[column]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("₦", "", regex=False)
        .str.strip(),
        errors="coerce"
    ).fillna(0)


# Remove invalid dates and arrange the months correctly
plot_data = (
    plot_data
    .dropna(subset=[month_column])
    .sort_values(month_column)
)


# Create the chart
fig, ax = plt.subplots(figsize=(15, 7))


# Plot income in green
ax.plot(
    plot_data[month_column],
    plot_data[income_column],
    color="green",
    marker="o",
    markersize=7,
    linewidth=2.5,
    label="Income"
)


# Plot outcome/spending in red
ax.plot(
    plot_data[month_column],
    plot_data[outcome_column],
    color="red",
    marker="o",
    markersize=7,
    linewidth=2.5,
    label="Outcome/Spending"
)


# Place an x-axis tick under every data point
ax.set_xticks(plot_data[month_column])


# Format dates as month and year
ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%b %Y")
)


# Add horizontal and vertical grid lines
ax.grid(
    True,
    which="major",
    axis="both",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5
)

# Keep the lines and points above the grid
ax.set_axisbelow(True)


# Format the y-axis as naira
ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda value, position: f"₦{value:,.0f}"
    )
)


# Add chart title and labels
ax.set_title(
    "Monthly Income and Spending",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Amount", fontsize=12)

ax.legend(fontsize=11)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()